In [14]:
import re
from collections import defaultdict
import pandas as pd
from IPython.display import display

RE_RECEIVED = re.compile(r"Received routine\s+([^\s]+)")
RE_READ     = re.compile(r"Read\s+(\d+)\s+bytes from the buffer")
RE_CALLED   = re.compile(r"Called:\s*([^\s]+)")

def stat_backend_log(log_file):
    stats = defaultdict(lambda: {"count": 0, "bytes": 0})
    pending_method = None
    orphan_reads = 0
    mismatched_called = 0

    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            # 收到 routine
            m = RE_RECEIVED.search(line)
            if m:
                pending_method = m.group(1)
                continue
            
            # 读取字节数
            m = RE_READ.search(line)
            if m:
                bytes_read = int(m.group(1))
                if pending_method:
                    stats[pending_method]["count"] += 1
                    stats[pending_method]["bytes"] += bytes_read
                    pending_method = None
                else:
                    orphan_reads += 1
                continue

            # 检查 Called
            m = RE_CALLED.search(line)
            if m and pending_method:
                called = m.group(1)
                if called != pending_method:
                    mismatched_called += 1
                pending_method = None
    return stats, orphan_reads, mismatched_called
    
RE_ROUTINE = re.compile(r"DEBUG - Routine '([^']+)' returned \d+")
RE_BUFFER_SIZE = re.compile(r"Output buffer size: (\d+)")
def stat_frontend_log(log_file):
    stats = defaultdict(lambda: {"count": 0, "bytes": 0})
    current_routine = None
    orphan_reads = 0

    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            
            # Match routine calls
            routine_match = RE_ROUTINE.search(line)
            if routine_match:
                current_routine = routine_match.group(1)
                continue
                
            # Match buffer sizes
            buffer_match = RE_BUFFER_SIZE.search(line)
            if buffer_match:
                bytes_read = int(buffer_match.group(1))
                
                if current_routine:
                    stats[current_routine]["count"] += 1
                    stats[current_routine]["bytes"] += bytes_read
                    current_routine = None
                else:
                    orphan_reads += 1
    return stats, orphan_reads

In [15]:
def display_stat(stats):
    
    df = pd.DataFrame(
        [
            {
                "CUDA Routine": method,
                "Calls": v["count"],
                "Total Bytes": v["bytes"],
                "Avg Bytes/Call": v["bytes"] / v["count"] if v["count"] else 0
            }
            for method, v in stats.items()
        ]
    ).sort_values(by="Calls", ascending=False).reset_index(drop=True)

    total_calls = df["Calls"].sum()
    total_bytes = df["Total Bytes"].sum()
    global_avg = total_bytes / total_calls if total_calls > 0 else 0

    summary_df = pd.DataFrame([{
            "CUDA Routine": "TOTAL",
            "Calls": total_calls,
            "Total Bytes": total_bytes,
            "Avg Bytes/Call": global_avg
        }])
    df = pd.concat([df, summary_df], ignore_index=True)
    pd.set_option('display.float_format', '{:,.2f}'.format)
    sort_column = "Calls"
    formatted_df = df.style\
        .format({
            sort_column: "{:,}",
            "Total Bytes": "{:,}",
            "Avg Bytes/Call": "{:,.2f}"
        })\
        .set_properties(**{'text-align': 'center'})\
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center')]},
            {'selector': '.row_heading, .blank', 'props': [('display', 'none;')]},
            {'selector': 'tr:last-child', 'props': [('font-weight', 'bold')]}
        ])

    return formatted_df

In [17]:


backend_log = "gvirtus_logs/Batch4000_100times/backend.log"  # 改成你的日志文件路径

# 統計資料
stats, orphan_reads, mismatched_called = stat_backend_log(backend_log)

# 转成 DataFrame
formatted_df = display_stat(stats)
display(formatted_df)

if orphan_reads or mismatched_called:
    print("\n[注意]")
    if orphan_reads:
        print(f"- 有 {orphan_reads} 次 'Read ... bytes' 没有匹配到 'Received routine'")
    if mismatched_called:
        print(f"- 有 {mismatched_called} 次 'Called:' 与对应 'Received routine' 方法名不一致")

log_file = "gvirtus_logs/Batch4000_100times/frontend.log"

# Optimized regex patterns
stats, orphan_reads = stat_frontend_log(log_file)

formatted_df = display_stat(stats)
display(formatted_df)

# Print diagnostics
if orphan_reads:
    print(f"\n[Note] Found {orphan_reads} buffer size readings without preceding routine call")


,CUDA Routine,Calls,Total Bytes,Avg Bytes/Call
0,cudaLaunchKernel,"9,000","708,000",78.67
1,cudaPushCallConfiguration,"9,000","360,000",40.00
2,cudaPopCallConfiguration,"9,000",0,0.00
3,cudaGetLastError,"9,000",0,0.00
4,cudaMemcpy,"1,006","12,544,037,540","12,469,222.21"
5,cudaStreamSynchronize,"1,002","8,016",8.00
6,cudaMemcpyAsync,"1,000","37,000",37.00
7,cudaRegisterFunction,9,"1,541",171.22
8,cudaFreeAsync,7,112,16.00
9,cudaMallocAsync,7,112,16.00


,CUDA Routine,Calls,Total Bytes,Avg Bytes/Call
0,cudaLaunchKernel,"9,000",0,0.00
1,cudaPushCallConfiguration,"9,000",0,0.00
2,cudaPopCallConfiguration,"9,000","360,000",40.00
3,cudaGetLastError,"9,000",0,0.00
4,cudaMemcpy,"1,006",0,0.00
5,cudaStreamSynchronize,"1,002",0,0.00
6,cudaMemcpyAsync,"1,000","160,008,000","160,008.00"
7,cudaRegisterFunction,9,757,84.11
8,cudaFreeAsync,7,0,0.00
9,cudaMallocAsync,7,56,8.00


In [16]:

backend_log = "gvirtus_logs/Batch4000_100times/backend_graph.log"  # 改成你的日志文件路径

# 統計資料
stats, orphan_reads, mismatched_called = stat_backend_log(backend_log)

# 转成 DataFrame
formatted_df = display_stat(stats)
display(formatted_df)

if orphan_reads or mismatched_called:
    print("\n[注意]")
    if orphan_reads:
        print(f"- 有 {orphan_reads} 次 'Read ... bytes' 没有匹配到 'Received routine'")
    if mismatched_called:
        print(f"- 有 {mismatched_called} 次 'Called:' 与对应 'Received routine' 方法名不一致")


log_file = "gvirtus_logs/Batch4000_100times/frontend_graph.log"

# Optimized regex patterns
stats, orphan_reads = stat_frontend_log(log_file)

formatted_df = display_stat(stats)
display(formatted_df)

# Print diagnostics
if orphan_reads:
    print(f"\n[Note] Found {orphan_reads} buffer size readings without preceding routine call")


,CUDA Routine,Calls,Total Bytes,Avg Bytes/Call
0,cudaMemcpyAsync,"2,000","12,544,073,000","6,272,036.50"
1,cudaStreamSynchronize,"1,002","8,016",8.00
2,cudaGraphLaunch,"1,001","16,016",16.00
3,cudaRegisterFunction,9,"1,541",171.22
4,cudaGetLastError,9,0,0.00
5,cudaLaunchKernel,9,708,78.67
6,cudaPopCallConfiguration,9,0,0.00
7,cudaPushCallConfiguration,9,360,40.00
8,cudaMallocAsync,7,112,16.00
9,cudaFreeAsync,7,112,16.00


,CUDA Routine,Calls,Total Bytes,Avg Bytes/Call
0,cudaMemcpyAsync,"2,000","160,008,000","80,004.00"
1,cudaStreamSynchronize,"1,002",0,0.00
2,cudaGraphLaunch,"1,001",0,0.00
3,cudaRegisterFunction,9,757,84.11
4,cudaGetLastError,9,0,0.00
5,cudaLaunchKernel,9,0,0.00
6,cudaPopCallConfiguration,9,360,40.00
7,cudaPushCallConfiguration,9,0,0.00
8,cudaMallocAsync,7,56,8.00
9,cudaFreeAsync,7,0,0.00


In [1]:
import re
from collections import defaultdict
import pandas as pd
from IPython.display import display

log_file = "gvirtus_logs/backend.log"

# Optimized regex patterns
RE_ROUTINE = re.compile(r"DEBUG - Routine '([^']+)' returned \d+")
RE_BUFFER_SIZE = re.compile(r"DEBUG - Output buffer size: (\d+)")

stats = defaultdict(lambda: {"count": 0, "bytes": 0})
current_routine = None
orphan_reads = 0

with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        
        # Match routine calls
        routine_match = RE_ROUTINE.search(line)
        if routine_match:
            current_routine = routine_match.group(1)
            continue
            
        # Match buffer sizes
        buffer_match = RE_BUFFER_SIZE.search(line)
        if buffer_match:
            bytes_read = int(buffer_match.group(1))
            if current_routine:
                stats[current_routine]["count"] += 1
                stats[current_routine]["bytes"] += bytes_read
                current_routine = None
            else:
                orphan_reads += 1

# Create DataFrame with consistent column names
column_map = {
    "Method": "CUDA Routine",
    "count": "Calls",
    "bytes": "Total Bytes",
    "AvgBytes": "Avg Bytes/Call"
}

try:
    df = pd.DataFrame(
        [
            {
                column_map["Method"]: method,
                column_map["count"]: v["count"],
                column_map["bytes"]: v["bytes"],
                column_map["AvgBytes"]: v["bytes"] / v["count"] if v["count"] else 0
            }
            for method, v in stats.items()
        ]
    )
    
    if not df.empty:
        # Verify sort column exists
        sort_column = column_map["count"]
        if sort_column in df.columns:
            df = df.sort_values(by=sort_column, ascending=False).reset_index(drop=True)
            
            # Calculate totals
            total_calls = df[sort_column].sum()
            total_bytes = df[column_map["bytes"]].sum()
            global_avg = total_bytes / total_calls if total_calls else 0
            
            # Add summary row
            summary_df = pd.DataFrame([{
                column_map["Method"]: "TOTAL",
                sort_column: total_calls,
                column_map["bytes"]: total_bytes,
                column_map["AvgBytes"]: global_avg
            }])
            
            df = pd.concat([df, summary_df], ignore_index=True)
            
            # Formatting
            pd.set_option('display.float_format', '{:,.2f}'.format)
            formatted_df = df.style\
                .format({
                    sort_column: "{:,}",
                    column_map["bytes"]: "{:,}",
                    column_map["AvgBytes"]: "{:,.2f}"
                })\
                .set_properties(**{'text-align': 'center'})\
                .set_table_styles([
                    {'selector': 'th', 'props': [('text-align', 'center')]},
                    {'selector': '.row_heading, .blank', 'props': [('display', 'none;')]},
                    {'selector': 'tr:last-child', 'props': [('font-weight', 'bold')]}
                ])
            
            display(formatted_df)
        else:
            print(f"Error: Column '{sort_column}' not found in DataFrame")
    else:
        print("No data found in log file")
        
except Exception as e:
    print(f"Error processing data: {str(e)}")
    if 'df' in locals():
        print("\nRaw DataFrame columns:", df.columns.tolist())

# Print diagnostics
if orphan_reads:
    print(f"\n[Note] Found {orphan_reads} buffer size readings without preceding routine call")

No data found in log file
